In [1]:
!pip install scikit-learn

In [2]:
import sys
!{sys.executable} -m pip install wfdb scikit-learn

  Using cached pycparser-3.0-py3-none-any.whl.metadata (8.2 kB)
   ---------------------------------------- 0.0/41.3 MB ? eta -:--:--
   ---------------------------------------- 0.3/41.3 MB ? eta -:--:--
    --------------------------------------- 0.5/41.3 MB 1.5 MB/s eta 0:00:27
    --------------------------------------- 0.8/41.3 MB 1.5 MB/s eta 0:00:27
   - -------------------------------------- 1.0/41.3 MB 1.5 MB/s eta 0:00:27
   - -------------------------------------- 1.3/41.3 MB 1.6 MB/s eta 0:00:26
   - -------------------------------------- 1.6/41.3 MB 1.5 MB/s eta 0:00:27
   - -------------------------------------- 1.8/41.3 MB 1.4 MB/s eta 0:00:28
   -- ------------------------------------- 2.1/41.3 MB 1.4 MB/s eta 0:00:29
   -- ------------------------------------- 2.4/41.3 MB 1.4 MB/s eta 0:00:29
   -- ------------------------------------- 2.6/41.3 MB 1.3 MB/s eta 0:00:29
   -- ------------------------------------- 2.9/41.3 MB 1.3 MB/s eta 0:00:29
   --- -------------------

In [3]:
# =====================================================================================
# Project: PTB-XL Metadata Audit  -  v0.4 GPU environment, signal pipeline, models,
#          timing/VRAM benchmark, and the write-once DEVELOPMENT PROTOCOL
# Code version: v0.4  (reviewer PATCH applied before first run: fusion location, epochs
#                      fixed at 50, two extra dev_protocol fields)
# Date: 2026-10-02
# Purpose: prepare everything the H3 signal experiments need, WITHOUT computing any
#          performance metric:
#            STEP 0  verify the sealed pre-registration (v0.3) and input CSV hashes;
#                    detect an existing dev_protocol.json
#            A  environment report (torch / CUDA / cuDNN / GPU / wfdb), seeds and
#               determinism settings
#            B  load 100 Hz signals for folds 1-9 ONLY, integrity checks, cache,
#               per-lead normalization parameters from folds 1-8, label alignment check
#            C  metadata matrices for the three arms (signal_only / +physio / +all)
#            D  resnet1d_wang (primary) and inception1d (robustness) in PyTorch, with
#               metadata fusion exactly as sealed; shape / gradient checks
#            E  one-epoch timing and peak-VRAM benchmark per architecture
#               (arm signal_plus_all, batch 128, 250-sample random crops) + projected
#               GPU-hour budget
#            F  write dev_protocol.json ONCE (exclusive create, self_sha256)
#
# THIS STAGE COMPUTES NO PERFORMANCE METRIC ON ANY DATA: no AUROC, no AUPRC, no loss on
# fold 9. Fold-9 signals are loaded for integrity checks only. Training-batch losses in
# Parts D/E are only asserted finite and are never printed or saved. Reason: the
# hyperparameter-selection rule (Part F) must be locked before any signal-model result
# is seen.
#
# FOLD 10: no fold-10 record file is ever opened. The list of files to open is built
# from rows with strat_fold <= 9 only and is asserted disjoint from fold 10's file list
# (derived from the CSV metadata, as in v0.2.1/v0.3).
#
# REUSE (verbatim from v0.3): config style, sha256_of, hdr/warn/note, _json_default,
# canonical_json_bytes, label derivation, metadata preprocessing (age sentinel,
# encode_categorical with __MISSING__/__UNKNOWN__). StandardScaler is fit on folds 1-8
# and applied to age_imputed only (recording_year is not used by any arm); everything
# else stays raw 0/1.
#
# Windows + Jupyter: DataLoader num_workers=0.
#
# Outputs:
#   OUT_DIR = D:\ptbxl-metadata-audit\outputs\v0.4
#     1) v0.4_env_report.json
#     2) v0.4_signals_folds1to9_100hz.npy + v0.4_signals_index.csv
#     3) v0.4_lead_norm_params.csv
#     4) v0.4_signal_integrity.csv
#     5) v0.4_model_summary.csv
#     6) v0.4_benchmark.csv + v0.4_budget_projection.csv
#     8) v0.4_manifest.json  (hashes of all outputs except itself)
#   PREREG_DIR = D:\ptbxl-metadata-audit\preregistration
#     7) dev_protocol.json   (write-once, exclusive create, self_sha256 inside)
# =====================================================================================

import os, sys, json, ast, hashlib, platform, time, random, math
from datetime import datetime
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_rows", 300)
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 220)

# ------------------------------- configuration ---------------------------------------
DATA_ROOT = r"D:\ptbxl-metadata-audit\data\ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3"
OUT_DIR   = r"D:\ptbxl-metadata-audit\outputs\v0.4"
PREREG_DIR = r"D:\ptbxl-metadata-audit\preregistration"
SEAL_PATH = os.path.join(PREREG_DIR, "pre_registration.json")
DEV_PROTOCOL_PATH = os.path.join(PREREG_DIR, "dev_protocol.json")
CODE_VERSION = "v0.4"
PREFIX = CODE_VERSION + "_"
SUPERCLASSES = ["NORM", "MI", "STTC", "CD", "HYP"]

EXPECTED_SEAL_SHA256 = "5183f54569f3358a5339b16a2c3c3be49f5c7798eabe6b0bb25946119c2ba469"
EXPECTED_INPUT_HASHES = {
    "ptbxl_database.csv": "7600de9c1b27d181d850b3c6038a35d7c3ddb6bb33b702e3a20252a6859d216b",
    "scp_statements.csv": "ad05b0b1fcae83bb1230755ad9cfc7c96f303feddc08a4a9ad5bdc9ca63bac8f",
}
# v0.2.1 Section 0b: fold-9 positives per superclass
EXPECTED_FOLD9_POSITIVES = {"NORM": 955, "MI": 540, "STTC": 528, "CD": 495, "HYP": 268}
EXPECTED_N_RECORDS_FOLDS1TO9 = 19601
# v0.2.1 Section 1: one-hot column counts (levels incl. __MISSING__ + __UNKNOWN__)
EXPECTED_ONEHOT_COUNTS = {"device": 13, "site": 52, "nurse": 14}

SEED = 42
FS = 100
N_SAMPLES = 1000
N_LEADS = 12
NUM_WORKERS = 0                   # Windows + Jupyter

# training recipe (reference defaults; locked in Part F)
CROP = 250                        # 2.5 s random crops for training
EVAL_STRIDE = 125                 # sliding windows for evaluation (not used in v0.4)
BENCH_BATCH_SIZE = 128
LR_GRID = [1e-3, 3e-3, 1e-2]
MAX_EPOCHS = 50
WEIGHT_DECAY = 1e-2
GPU_BUDGET_HOURS = 30.0
N_SEEDS = 3

ARCHS = ["resnet1d_wang", "inception1d"]
PRIMARY_ARCH = "resnet1d_wang"
ROBUSTNESS_ARCH = "inception1d"
ARMS = ["signal_only", "signal_plus_physio", "signal_plus_all"]
EXPECTED_META_DIM = {"signal_only": 0, "signal_plus_physio": 3, "signal_plus_all": 82}

SIGNALS_NPY = os.path.join(OUT_DIR, PREFIX + "signals_folds1to9_100hz.npy")
SIGNALS_INDEX = os.path.join(OUT_DIR, PREFIX + "signals_index.csv")

_root_abs = os.path.abspath(DATA_ROOT)
for _p in (OUT_DIR, PREREG_DIR):
    assert not os.path.abspath(_p).lower().startswith(_root_abs.lower()), \
        "output path is inside the dataset folder - aborting: " + _p

RUN_STARTED = datetime.now().astimezone().isoformat()
OUTPUT_FILES = []        # paths of produced files (for the manifest)
WARNINGS = []


def sha256_of(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()


def save_csv(obj, name, index=True):
    p = os.path.join(OUT_DIR, name)
    obj.to_csv(p, index=index, encoding="utf-8-sig")
    OUTPUT_FILES.append(p)
    print("[saved] " + p)
    return p


def hdr(t):
    print("\n" + "=" * 100)
    print(t)
    print("=" * 100)


def warn(msg):
    WARNINGS.append(msg)
    print("[WARN] " + msg)


def note(msg):
    print("[NOTE] " + msg)


def _json_default(o):
    if isinstance(o, np.integer):
        return int(o)
    if isinstance(o, np.floating):
        return float(o)
    if isinstance(o, np.bool_):
        return bool(o)
    if isinstance(o, np.ndarray):
        return o.tolist()
    return str(o)


def to_level_string(series, missing="__MISSING__"):
    """Human-readable, zero-padded categorical level for numeric columns (e.g.
    '007' for site) so sorting and printed feature names stay legible."""
    if pd.api.types.is_numeric_dtype(series):
        nn = series.dropna()
        integral = bool(len(nn) == 0 or np.all(np.isclose(nn.values, np.round(nn.values))))
        if integral:
            width = max(3, len(str(int(nn.max()))) if len(nn) else 3)
            return series.apply(lambda v: missing if pd.isna(v) else str(int(v)).zfill(width))
    return series.apply(lambda v: missing if pd.isna(v) else str(v).strip())


def encode_categorical(level_series, train_mask, col_name):
    """One-hot encode a pre-stringified categorical column. The category list is
    fixed from the training rows only, always includes an explicit '__MISSING__'
    level, and always includes a catch-all '__UNKNOWN__' level for anything seen
    only outside training (e.g. only in fold 9). Both splits therefore produce the
    exact same set of columns."""
    train_vals = level_series[train_mask]
    train_categories = sorted(set(train_vals.unique().tolist()) | {"__MISSING__"})
    known = set(train_categories)
    remapped = level_series.where(level_series.isin(known), other="__UNKNOWN__")
    categories = train_categories + ["__UNKNOWN__"]
    cat = pd.Categorical(remapped, categories=categories)
    dummies = pd.get_dummies(cat)
    dummies.columns = ["{}={}".format(col_name, c) for c in dummies.columns]
    dummies.index = level_series.index
    return dummies, categories


def canonical_json_bytes(d):
    """Canonical serialization used for self_sha256 (see SELF_SHA256_PROCEDURE)."""
    return json.dumps(d, sort_keys=True, ensure_ascii=False, separators=(",", ":"),
                      default=_json_default).encode("utf-8")


SELF_SHA256_PROCEDURE = (
    "Load the JSON file; remove the key 'self_sha256' (the key must be absent, not empty); "
    "serialize the remaining dict with json.dumps(d, sort_keys=True, ensure_ascii=False, "
    "separators=(',', ':')); compute SHA-256 of the UTF-8 bytes; the hex digest must equal "
    "the stored 'self_sha256'. (At write time numpy scalars were converted to plain JSON "
    "types first, so the loaded file hashes identically.)"
)


def verify_self_sha256(path):
    """Return (stored, recomputed, loaded_dict) for a self-hashed JSON document."""
    with open(path, "r", encoding="utf-8") as f:
        d = json.load(f)
    stored = d.pop("self_sha256", None)
    recomputed = hashlib.sha256(canonical_json_bytes(d)).hexdigest()
    d["self_sha256"] = stored
    return stored, recomputed, d


# =====================================================================================
# STEP 0 - SEAL INTEGRITY (before anything else; no file is written before this passes)
# =====================================================================================
hdr("STEP 0 - SEAL INTEGRITY")

assert os.path.isfile(SEAL_PATH), "sealed pre-registration not found: " + SEAL_PATH + " - aborting."
_seal_stored, _seal_recomputed, SEAL = verify_self_sha256(SEAL_PATH)
print("pre_registration.json stored self_sha256    :", _seal_stored)
print("pre_registration.json recomputed self_sha256:", _seal_recomputed)
print("expected (v0.3 output)                      :", EXPECTED_SEAL_SHA256)
assert _seal_stored == _seal_recomputed, "SEAL BROKEN: stored self_sha256 != recomputed - aborting."
assert _seal_stored == EXPECTED_SEAL_SHA256, "SEAL MISMATCH: self_sha256 != v0.3 constant - aborting."
assert SEAL.get("sealed") is True, "pre_registration.json is not marked sealed - aborting."
print("seal integrity: PASS")

print("\n--- H3_protocol (from the sealed file; it governs everything below) ---")
print(json.dumps(SEAL["H3_protocol"], ensure_ascii=False, indent=2))

DB_CSV = os.path.join(DATA_ROOT, "ptbxl_database.csv")
SCP_CSV = os.path.join(DATA_ROOT, "scp_statements.csv")
for p in (DB_CSV, SCP_CSV):
    assert os.path.isfile(p), "file not found: " + p
INPUT_HASHES = {os.path.basename(p): sha256_of(p) for p in (DB_CSV, SCP_CSV)}
print()
for k, v in INPUT_HASHES.items():
    match = (v == EXPECTED_INPUT_HASHES[k])
    print("{:26s} {}  matches v0.3: {}".format(k, v, "YES" if match else "NO"))
    assert match, "input file hash differs from v0.3 for " + k + " - aborting."
assert INPUT_HASHES == SEAL["input_files_sha256"], "input hashes differ from those inside the seal - aborting."

DEV_PROTOCOL_EXISTS = os.path.exists(DEV_PROTOCOL_PATH)
DEV_PROTOCOL_SHA = None
if DEV_PROTOCOL_EXISTS:
    _dp_stored, _dp_recomputed, _dp = verify_self_sha256(DEV_PROTOCOL_PATH)
    assert _dp_stored == _dp_recomputed, "dev_protocol.json BROKEN: stored != recomputed self_sha256 - aborting."
    assert _dp.get("parent_seal_self_sha256") == EXPECTED_SEAL_SHA256, \
        "dev_protocol.json points to a different parent seal - aborting."
    DEV_PROTOCOL_SHA = _dp_stored
    print("\ndev_protocol.json ALREADY EXISTS (self_sha256={}, verified) -> Part F will be "
          "SKIPPED; nothing will be overwritten. Contents:".format(_dp_stored))
    print(json.dumps(_dp, ensure_ascii=False, indent=2))
else:
    print("\ndev_protocol.json not present -> Part F will write it once (exclusive create).")

os.makedirs(OUT_DIR, exist_ok=True)


# =====================================================================================
# PART A - ENVIRONMENT REPORT
# =====================================================================================
hdr("PART A - ENVIRONMENT REPORT")

try:
    import torch
    import torch.nn as nn
    from torch.utils.data import Dataset, DataLoader
except ImportError as e:
    raise RuntimeError("PyTorch is not installed in this kernel ({}). Install a CUDA build of "
                       "torch into the 'medssl' environment and re-run.".format(e))
try:
    import wfdb
except ImportError as e:
    raise RuntimeError("wfdb is not installed in this kernel ({}). Run: pip install wfdb".format(e))
import sklearn

if not torch.cuda.is_available():
    raise RuntimeError("torch.cuda.is_available() is False: no usable CUDA GPU in this kernel "
                       "(torch {}, built for CUDA {}). Install a CUDA-enabled torch build or fix the "
                       "driver, then re-run. Aborting.".format(torch.__version__, torch.version.cuda))
DEVICE = torch.device("cuda:0")


def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_all_seeds(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

_props = torch.cuda.get_device_properties(0)
ENV = {
    "python": sys.version.replace("\n", " "),
    "os": platform.platform(),
    "numpy": np.__version__, "pandas": pd.__version__, "sklearn": sklearn.__version__,
    "torch": torch.__version__,
    "torch_cuda": torch.version.cuda,
    "cudnn": torch.backends.cudnn.version(),
    "gpu_name": torch.cuda.get_device_name(0),
    "gpu_total_vram_gb": round(_props.total_memory / 1024 ** 3, 3),
    "wfdb": getattr(wfdb, "__version__", "unknown"),
    "determinism": {
        "seed": SEED,
        "seeded": ["python random", "numpy", "torch", "torch.cuda (all devices)"],
        "cudnn_deterministic": torch.backends.cudnn.deterministic,
        "cudnn_benchmark": torch.backends.cudnn.benchmark,
        "torch_use_deterministic_algorithms": False,
        "note": ("cudnn.deterministic=True and cudnn.benchmark=False; "
                 "torch.use_deterministic_algorithms is NOT enabled, so a few CUDA kernels "
                 "(e.g. some pooling backward passes) may remain non-bitwise-reproducible"),
    },
    "dataloader_num_workers": NUM_WORKERS,
}
for k, v in ENV.items():
    if k != "determinism":
        print("{:22s}: {}".format(k, v))
print("determinism           :", json.dumps(ENV["determinism"]["seeded"]),
      "| cudnn.deterministic=True | cudnn.benchmark=False")

ENV_PATH = os.path.join(OUT_DIR, PREFIX + "env_report.json")
with open(ENV_PATH, "w", encoding="utf-8") as f:
    json.dump(ENV, f, ensure_ascii=False, indent=2, default=_json_default)
OUTPUT_FILES.append(ENV_PATH)
print("[saved] " + ENV_PATH)


# =====================================================================================
# PART B - SIGNAL LOADING (folds 1-9 only, 100 Hz)
# =====================================================================================
hdr("PART B - SIGNAL LOADING (folds 1-9 only, 100 Hz)")

df_all = pd.read_csv(DB_CSV, index_col="ecg_id")
df9 = df_all[df_all["strat_fold"] <= 9].copy()
mask_train = df9["strat_fold"] <= 8
mask_eval = df9["strat_fold"] == 9
assert (df9["strat_fold"] == 10).sum() == 0, "fold 10 leaked into the working set - aborting."
n_fold10_in_source = int((df_all["strat_fold"] == 10).sum())

train_patients = set(df9.loc[mask_train, "patient_id"])
eval_patients = set(df9.loc[mask_eval, "patient_id"])
assert train_patients.isdisjoint(eval_patients), \
    "patient-level leakage between train (folds 1-8) and eval (fold 9) - aborting."

FILES_TO_OPEN = df9["filename_lr"].tolist()
_fold10_files = set(df_all.loc[df_all["strat_fold"] == 10, "filename_lr"].tolist())
assert len(FILES_TO_OPEN) == EXPECTED_N_RECORDS_FOLDS1TO9, \
    "expected {} records in folds 1-9, got {}".format(EXPECTED_N_RECORDS_FOLDS1TO9, len(FILES_TO_OPEN))
assert _fold10_files.isdisjoint(FILES_TO_OPEN), "a fold-10 record file is in the open-list - aborting."
del _fold10_files
print("files to open: {} (folds 1-9) | fold-10 files in list: 0 (asserted) | fold-10 records in "
      "source CSV, never opened: {}".format(len(FILES_TO_OPEN), n_fold10_in_source))

current_index = pd.DataFrame({
    "ecg_id": df9.index.to_numpy(),
    "strat_fold": df9["strat_fold"].to_numpy(),
    "patient_id": df9["patient_id"].to_numpy(),
})
N_REC = len(current_index)


def _record_path(fn):
    return os.path.join(DATA_ROOT, fn)


cache_ok = False
if os.path.isfile(SIGNALS_NPY) and os.path.isfile(SIGNALS_INDEX):
    _old_idx = pd.read_csv(SIGNALS_INDEX, encoding="utf-8-sig")
    if (list(_old_idx.columns) == list(current_index.columns)
            and _old_idx.astype(str).reset_index(drop=True).equals(
                current_index.astype(str).reset_index(drop=True))):
        cache_ok = True

N_FILES_OPENED = 0
if cache_ok:
    print("cache found and its sidecar index matches exactly -> reusing " + SIGNALS_NPY)
    X = np.load(SIGNALS_NPY)
    assert X.shape == (N_REC, N_LEADS, N_SAMPLES) and X.dtype == np.float32, \
        "cached array has unexpected shape/dtype {} {} - delete the cache and re-run.".format(X.shape, X.dtype)
    # lead order / fs from the header of the first (fold 1-9) record only
    _h = wfdb.rdheader(_record_path(FILES_TO_OPEN[0]))
    assert int(_h.fs) == FS, "fs != 100 in header of " + FILES_TO_OPEN[0]
    LEAD_NAMES = list(_h.sig_name)
    note("on cache reuse, fs/shape/lead-order were verified for every record when the cache was "
         "built; this run re-reads only the first record's header.")
else:
    print("no valid cache -> reading {} records with wfdb.rdsamp ...".format(N_REC))
    X = np.empty((N_REC, N_LEADS, N_SAMPLES), dtype=np.float32)
    LEAD_NAMES = None
    t0 = time.time()
    for i, fn in enumerate(FILES_TO_OPEN):
        sig, fields = wfdb.rdsamp(_record_path(fn))
        N_FILES_OPENED += 1
        assert int(fields["fs"]) == FS, "fs != 100 for " + fn
        assert sig.shape == (N_SAMPLES, N_LEADS), "shape {} != (1000, 12) for {}".format(sig.shape, fn)
        names = list(fields["sig_name"])
        if LEAD_NAMES is None:
            LEAD_NAMES = names
        assert names == LEAD_NAMES, "lead names/order differ for {}: {}".format(fn, names)
        X[i] = sig.T.astype(np.float32)
        if (i + 1) % 2000 == 0 or i + 1 == N_REC:
            print("  read {:6d}/{} records ({:.0f}s)".format(i + 1, N_REC, time.time() - t0))
    assert N_FILES_OPENED == EXPECTED_N_RECORDS_FOLDS1TO9
    _tmp = SIGNALS_NPY + ".tmp"
    with open(_tmp, "wb") as f:
        np.save(f, X)
    os.replace(_tmp, SIGNALS_NPY)
    current_index.to_csv(SIGNALS_INDEX, index=False, encoding="utf-8-sig")
    print("[saved] " + SIGNALS_NPY)
    print("[saved] " + SIGNALS_INDEX)
OUTPUT_FILES += [SIGNALS_NPY, SIGNALS_INDEX]
print("record files opened this run: {} | X shape {} {} | lead order: {}".format(
    N_FILES_OPENED, X.shape, X.dtype, LEAD_NAMES))

# --- integrity: NaN/Inf (must be 0), all-zero leads, flat leads (std == 0), per fold ---
folds_arr = current_index["strat_fold"].to_numpy()
integrity_rows = []
CHUNK = 1000
for fold in range(1, 10):
    rows = np.where(folds_arr == fold)[0]
    r = {"strat_fold": fold, "n_records": len(rows), "n_nan_values": 0, "n_inf_values": 0,
         "n_records_with_nan_or_inf": 0, "n_allzero_leads": 0, "n_records_with_allzero_lead": 0,
         "n_flat_leads": 0, "n_records_with_flat_lead": 0}
    for s in range(0, len(rows), CHUNK):
        xc = X[rows[s:s + CHUNK]]
        nan_rec = np.isnan(xc).sum(axis=(1, 2))
        inf_rec = np.isinf(xc).sum(axis=(1, 2))
        allzero = (xc == 0).all(axis=2)
        flat = (xc.std(axis=2) == 0)
        r["n_nan_values"] += int(nan_rec.sum())
        r["n_inf_values"] += int(inf_rec.sum())
        r["n_records_with_nan_or_inf"] += int(((nan_rec + inf_rec) > 0).sum())
        r["n_allzero_leads"] += int(allzero.sum())
        r["n_records_with_allzero_lead"] += int(allzero.any(axis=1).sum())
        r["n_flat_leads"] += int(flat.sum())
        r["n_records_with_flat_lead"] += int(flat.any(axis=1).sum())
    integrity_rows.append(r)
integrity_df = pd.DataFrame(integrity_rows)
_tot = integrity_df.drop(columns=["strat_fold"]).sum()
_tot["strat_fold"] = "__TOTAL_1to9__"
integrity_df = pd.concat([integrity_df, pd.DataFrame([_tot])], ignore_index=True)
print("\n--- signal integrity per fold ---")
print(integrity_df.to_string(index=False))
save_csv(integrity_df, PREFIX + "signal_integrity.csv", index=False)
assert int(_tot["n_nan_values"]) == 0 and int(_tot["n_inf_values"]) == 0, \
    "NaN/Inf values present in the signals - aborting (see v0.4_signal_integrity.csv)."
if int(_tot["n_flat_leads"]) > 0:
    note("{} flat leads (std == 0) in {} records, of which {} all-zero leads; reported and KEPT "
         "(nothing is dropped).".format(int(_tot["n_flat_leads"]), int(_tot["n_records_with_flat_lead"]),
                                       int(_tot["n_allzero_leads"])))
else:
    print("no flat or all-zero leads.")

# --- per-lead normalization parameters: folds 1-8 ONLY (two-pass, float64) ---
train_rows = np.where(mask_train.to_numpy())[0]
eval_rows = np.where(mask_eval.to_numpy())[0]
_sum = np.zeros(N_LEADS, dtype=np.float64)
for s in range(0, len(train_rows), CHUNK):
    _sum += X[train_rows[s:s + CHUNK]].astype(np.float64).sum(axis=(0, 2))
_n_vals = len(train_rows) * N_SAMPLES
LEAD_MEAN = _sum / _n_vals
_sq = np.zeros(N_LEADS, dtype=np.float64)
for s in range(0, len(train_rows), CHUNK):
    xc = X[train_rows[s:s + CHUNK]].astype(np.float64) - LEAD_MEAN[None, :, None]
    _sq += (xc ** 2).sum(axis=(0, 2))
LEAD_STD = np.sqrt(_sq / _n_vals)
assert np.all(LEAD_STD > 0), "a lead has zero std over folds 1-8 - aborting."
norm_df = pd.DataFrame({"lead_index": np.arange(N_LEADS), "lead": LEAD_NAMES,
                        "mean": LEAD_MEAN, "std": LEAD_STD, "n_values": _n_vals,
                        "computed_on": "folds 1-8 only"})
print("\n--- per-lead normalization parameters (folds 1-8 only; not applied to any evaluated data) ---")
print(norm_df.round(6).to_string(index=False))
save_csv(norm_df, PREFIX + "lead_norm_params.csv", index=False)
LEAD_MEAN_F32 = LEAD_MEAN.astype(np.float32)[:, None]
LEAD_STD_F32 = LEAD_STD.astype(np.float32)[:, None]

# --- label derivation (verbatim from v0.2.1) + alignment check ---
scp = pd.read_csv(SCP_CSV, index_col=0)
diag = scp[scp["diagnostic"].fillna(0) == 1]
code2class = {k: v for k, v in diag["diagnostic_class"].items() if isinstance(v, str)}
parsed = df9["scp_codes"].apply(ast.literal_eval)
rec_classes = parsed.apply(lambda d: {code2class[k] for k in d if k in code2class})
lab = pd.DataFrame({c: rec_classes.apply(lambda s, c=c: int(c in s)) for c in SUPERCLASSES},
                   index=df9.index)
assert (lab.index == df9.index).all() and (current_index["ecg_id"].to_numpy() == lab.index.to_numpy()).all(), \
    "label rows are not aligned with the signal rows - aborting."
fold9_pos = {c: int(lab.loc[mask_eval, c].sum()) for c in SUPERCLASSES}
print("\nfold-9 positives:", fold9_pos, "| expected (v0.2.1):", EXPECTED_FOLD9_POSITIVES)
assert fold9_pos == EXPECTED_FOLD9_POSITIVES, "fold-9 label counts differ from v0.2.1 - aborting."
print("label alignment: PASS")
Y = lab[SUPERCLASSES].to_numpy(dtype=np.float32)


# =====================================================================================
# PART C - METADATA MATRICES FOR THE THREE ARMS (encoder fit on folds 1-8)
# =====================================================================================
hdr("PART C - METADATA MATRICES")

age_raw = df9["age"]
age_sentinel_mask = (age_raw == 300)
n_age_sentinel = int(age_sentinel_mask.sum())
age_clean = age_raw.where(~age_sentinel_mask, np.nan)
age_censored = age_sentinel_mask.astype(float)
age_imputed = age_clean.fillna(90.0)
sex = df9["sex"].astype(float)
print("age: sentinel 300 affects", n_age_sentinel, "records; flagged via age_censored, imputed with 90.")

device_level = to_level_string(df9["device"])
site_level = to_level_string(df9["site"])
nurse_level = to_level_string(df9["nurse"])
device_dummies, device_categories = encode_categorical(device_level, mask_train, "device")
site_dummies, site_categories = encode_categorical(site_level, mask_train, "site")
nurse_dummies, nurse_categories = encode_categorical(nurse_level, mask_train, "nurse")
for name, dmy in [("device", device_dummies), ("site", site_dummies), ("nurse", nurse_dummies)]:
    assert dmy.shape[1] == EXPECTED_ONEHOT_COUNTS[name], \
        "{} one-hot width {} != v0.2.1's {}".format(name, dmy.shape[1], EXPECTED_ONEHOT_COUNTS[name])

age_scaler = StandardScaler().fit(age_imputed[mask_train].to_numpy().reshape(-1, 1))
age_imputed_std = pd.Series(age_scaler.transform(age_imputed.to_numpy().reshape(-1, 1)).ravel(),
                            index=df9.index, name="age_imputed")

B1_cols = ["age_imputed", "age_censored", "sex"]
B2_cols = list(device_dummies.columns) + list(site_dummies.columns) + list(nurse_dummies.columns)
B3_cols = B1_cols + B2_cols

META_FEATURES = pd.concat([age_imputed_std, age_censored.rename("age_censored"), sex.rename("sex"),
                           device_dummies, site_dummies, nurse_dummies], axis=1)
ARM_COLUMNS = {"signal_only": [], "signal_plus_physio": B1_cols, "signal_plus_all": B3_cols}
META = {}
for arm in ARMS:
    cols = ARM_COLUMNS[arm]
    if cols:
        META[arm] = META_FEATURES[cols].to_numpy(dtype=np.float32)
    else:
        META[arm] = np.zeros((N_REC, 0), dtype=np.float32)
    assert META[arm].shape == (N_REC, EXPECTED_META_DIM[arm]), \
        "{}: metadata dim {} != {}".format(arm, META[arm].shape[1], EXPECTED_META_DIM[arm])
    assert np.isfinite(META[arm]).all()
    print("  {:20s} metadata dim = {:3d}{}".format(arm, META[arm].shape[1],
          "" if not cols else "  ({} ... {})".format(", ".join(cols[:3]), cols[-1])))
assert ARM_COLUMNS["signal_plus_physio"] == B1_cols and ARM_COLUMNS["signal_plus_all"] == B1_cols + B2_cols
print("column order: physio == v0.2.1 B1_physio, all == v0.2.1 B3_physio_plus_acquisition: PASS")
print("age_imputed scaler (folds 1-8): mean={:.4f}, scale={:.4f}".format(
    float(age_scaler.mean_[0]), float(age_scaler.scale_[0])))
FINAL_RUN_REFIT_NOTE = ("For the final run (week 8) the categorical encoder, the age_imputed scaler and the "
                        "per-lead signal normalization are refit on folds 1-9, as the sealed protocol "
                        "states (encoder fit on training folds only). v0.4 fits them on folds 1-8.")
note(FINAL_RUN_REFIT_NOTE)


# =====================================================================================
# PART D - MODEL IMPLEMENTATIONS (PyTorch)
# =====================================================================================
hdr("PART D - MODEL IMPLEMENTATIONS")

# >>> MODELS
# SOURCE FOLLOWED: Strodthoff, Wagner, Schaeffter, Samek (2021) "Deep Learning for ECG
# Analysis: Benchmarks and Insights from PTB-XL", reference code repository
# helme/ecg_ptbxl_benchmarking: code/models/resnet1d.py (resnet1d_wang, BasicBlock1d,
# ResNet1d), code/models/inception1d.py (Inception1d, InceptionBlock1d, Shortcut1d),
# code/models/basic_conv1d.py (create_head1d, bn_drop_lin, AdaptiveConcatPool1d), and
# code/models/fastai_model.py (how the benchmark instantiates them). Re-implemented here
# from memory of that code in plain PyTorch (no fastai); the repository was NOT
# consulted at write time.
#
# DETAILS I AM UNSURE OF (should be checked against the repository):
#  1) resnet1d_wang kernel sizes: factory default kernel_size=[5,3], stem kernel 7,
#     stride_stem=1, no stem pooling, inplanes=128, layers [1,1,1], fix_feature_dim=True
#     (all stages 128 channels), stride 2 in stages 2 and 3. If fastai_model.py passes
#     the int kernel_size=5, BasicBlock1d expands it to [5, 5//2+1] = [5,3] -> same.
#  2) inception1d as used by the benchmark: kernel_size = 8*5 = 40 -> branch kernels
#     [39,19,9], depth 6, bottleneck 32, nb_filters 32 (output 4*32 = 128 channels),
#     residual shortcut every 3 blocks; all convs bias=False.
#  3) Head (create_head1d) with lin_ftrs_head=[128], ps_head=0.5, bn=True, relu,
#     concat pooling: AdaptiveConcatPool1d [max, avg] -> 2*nf -> BN, Dropout(0.25),
#     Linear(2*nf,128), ReLU -> BN, Dropout(0.5), Linear(128, 5). I believe the
#     benchmark default is lin_ftrs_head=[128]; with lin_ftrs_head=None the head would be
#     a single Linear(2*nf, 5) instead.
#  4) Order inside AdaptiveConcatPool1d is [max, avg] (fastai convention).
#  5) The reference aggregates sliding-window predictions with aggregate_fn="max" by
#     default in fastai_model.py (I am fairly but not fully sure); the project's dev
#     protocol uses the MEAN over windows (Part F). Flagged, not changed.
#  6) MaxPool1d(3, stride, padding=1) on the inception max-pool branch.
#
# METADATA FUSION (v0.4 PATCH FIX 1 -- sealed text is binding: "concatenated to the
# network's global-pooled embedding before the final linear classification head").
# The global-pooled embedding is the 2*nf = 256-dim AdaptiveConcatPool1d output; the
# "head" is the whole MLP head. The head's first block keeps the reference order
# (BN, Dropout(0.25), Linear, ReLU), split as: BN + Dropout applied to the embedding
# (`pool_norm`, identical across arms), then the RAW metadata (no BN, no dropout) is
# concatenated, then Linear(256 + meta_dim, 128) -> ReLU -> BN -> Dropout(0.5) ->
# Linear(128, 5). Backbone, pooling and pool_norm are identical across arms; the head's
# first Linear differs in input width by design.


def conv1d_same(in_planes, out_planes, kernel_size=3, stride=1):
    return nn.Conv1d(in_planes, out_planes, kernel_size=kernel_size, stride=stride,
                     padding=(kernel_size - 1) // 2, bias=False)


class BasicBlock1d(nn.Module):
    expansion = 1

    def __init__(self, inplanes, planes, stride=1, kernel_size=(3, 3), downsample=None):
        super().__init__()
        if isinstance(kernel_size, int):
            kernel_size = [kernel_size, kernel_size // 2 + 1]
        self.conv1 = conv1d_same(inplanes, planes, kernel_size=kernel_size[0], stride=stride)
        self.bn1 = nn.BatchNorm1d(planes)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = conv1d_same(planes, planes, kernel_size=kernel_size[1])
        self.bn2 = nn.BatchNorm1d(planes)
        self.downsample = downsample

    def forward(self, x):
        residual = x if self.downsample is None else self.downsample(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return self.relu(out + residual)


def resnet1d_wang_backbone(input_channels=12, inplanes=128, kernel_size=(5, 3), kernel_size_stem=7,
                           stride_stem=1, layers=(1, 1, 1), stride=2):
    mods = [nn.Conv1d(input_channels, inplanes, kernel_size=kernel_size_stem, stride=stride_stem,
                      padding=(kernel_size_stem - 1) // 2, bias=False),
            nn.BatchNorm1d(inplanes), nn.ReLU(inplace=True)]          # no stem pooling
    cur = inplanes
    for i, n_blocks in enumerate(layers):
        st = 1 if i == 0 else stride
        planes = inplanes                                             # fix_feature_dim=True
        downsample = None
        if st != 1 or cur != planes:
            downsample = nn.Sequential(nn.Conv1d(cur, planes, kernel_size=1, stride=st, bias=False),
                                       nn.BatchNorm1d(planes))
        blocks = [BasicBlock1d(cur, planes, st, list(kernel_size), downsample)]
        cur = planes
        for _ in range(1, n_blocks):
            blocks.append(BasicBlock1d(cur, planes))
        mods.append(nn.Sequential(*blocks))
    return nn.Sequential(*mods), cur


class InceptionBlock1d(nn.Module):
    def __init__(self, ni, nb_filters, kss, stride=1, bottleneck_size=32):
        super().__init__()
        self.bottleneck = conv1d_same(ni, bottleneck_size, 1, stride) if bottleneck_size > 0 else nn.Identity()
        bi = bottleneck_size if bottleneck_size > 0 else ni
        self.convs = nn.ModuleList([conv1d_same(bi, nb_filters, ks) for ks in kss])
        self.conv_bottle = nn.Sequential(nn.MaxPool1d(3, stride, padding=1), conv1d_same(ni, nb_filters, 1))
        self.bn_relu = nn.Sequential(nn.BatchNorm1d((len(kss) + 1) * nb_filters), nn.ReLU())

    def forward(self, x):
        b = self.bottleneck(x)
        return self.bn_relu(torch.cat([c(b) for c in self.convs] + [self.conv_bottle(x)], dim=1))


class Shortcut1d(nn.Module):
    def __init__(self, ni, nf):
        super().__init__()
        self.act_fn = nn.ReLU(True)
        self.conv = conv1d_same(ni, nf, 1)
        self.bn = nn.BatchNorm1d(nf)

    def forward(self, inp, out):
        return self.act_fn(out + self.bn(self.conv(inp)))


class InceptionBackbone(nn.Module):
    def __init__(self, input_channels=12, kernel_size=40, depth=6, bottleneck_size=32, nb_filters=32,
                 use_residual=True):
        super().__init__()
        assert kernel_size >= 40 and depth % 3 == 0
        kss = [k - 1 if k % 2 == 0 else k for k in [kernel_size, kernel_size // 2, kernel_size // 4]]
        n_ks = len(kss) + 1
        self.depth, self.use_residual = depth, use_residual
        self.im = nn.ModuleList([InceptionBlock1d(input_channels if d == 0 else n_ks * nb_filters,
                                                  nb_filters, kss, bottleneck_size=bottleneck_size)
                                 for d in range(depth)])
        self.sk = nn.ModuleList([Shortcut1d(input_channels if d == 0 else n_ks * nb_filters, n_ks * nb_filters)
                                 for d in range(depth // 3)])
        self.out_channels = n_ks * nb_filters
        self.kss = kss

    def forward(self, x):
        input_res = x
        for d in range(self.depth):
            x = self.im[d](x)
            if self.use_residual and d % 3 == 2:
                x = self.sk[d // 3](input_res, x)
                input_res = x
        return x


class AdaptiveConcatPool1d(nn.Module):
    def __init__(self):
        super().__init__()
        self.mp = nn.AdaptiveMaxPool1d(1)
        self.ap = nn.AdaptiveAvgPool1d(1)

    def forward(self, x):
        return torch.cat([self.mp(x), self.ap(x)], dim=1).flatten(1)


class ECGNet(nn.Module):
    """Backbone + reference head. Metadata (meta_dim >= 0) is concatenated, raw, to the
    global-pooled embedding (after its BN + Dropout) and enters the head's first Linear.
    backbone / pool / pool_norm are built in the same order with the same shapes for
    every arm, so with the same seed they are initialised identically (checked below)."""

    def __init__(self, arch, meta_dim, n_classes=5, input_channels=12, lin_ftrs=128, ps_head=0.5):
        super().__init__()
        if arch == "resnet1d_wang":
            self.backbone, nf = resnet1d_wang_backbone(input_channels=input_channels)
        elif arch == "inception1d":
            self.backbone = InceptionBackbone(input_channels=input_channels, kernel_size=40)
            nf = self.backbone.out_channels
        else:
            raise ValueError(arch)
        self.pool = AdaptiveConcatPool1d()
        # embedding pre-processing: identical across arms
        self.pool_norm = nn.Sequential(nn.BatchNorm1d(2 * nf), nn.Dropout(ps_head / 2))
        self.embedding_dim = 2 * nf
        self.meta_dim = meta_dim
        # head: receives [embedding (+ raw metadata)]
        self.hidden = nn.Sequential(nn.Linear(2 * nf + meta_dim, lin_ftrs), nn.ReLU(inplace=True),
                                    nn.BatchNorm1d(lin_ftrs), nn.Dropout(ps_head))
        self.final = nn.Linear(lin_ftrs, n_classes)

    def forward(self, x, meta=None):
        z = self.pool_norm(self.pool(self.backbone(x)))
        if self.meta_dim > 0:
            z = torch.cat([z, meta], dim=1)   # raw metadata: no BN, no dropout
        return self.final(self.hidden(z))
# <<< MODELS


METADATA_FUSION_DESCRIPTION = (
    "location: the metadata vector is concatenated to the network's global-pooled embedding (the "
    "2*nf = 256-dim AdaptiveConcatPool1d [max, avg] output, after that embedding's BatchNorm1d + "
    "Dropout(0.25)) and the concatenation is the input of the head's first Linear "
    "(256 + meta_dim -> 128), followed by ReLU, BatchNorm1d(128), Dropout(0.5), Linear(128 -> 5); "
    "metadata enters RAW (age_imputed standardized with training-fold statistics, all other columns "
    "0/1): no BatchNorm and no dropout is applied to the metadata; identical across arms: backbone, "
    "global pooling and the embedding's BatchNorm + Dropout (same seed -> identical initial weights, "
    "asserted in v0.4); differs across arms by design: only the input width of the head's first "
    "Linear (256 / 259 / 338 for signal_only / signal_plus_physio / signal_plus_all)")
SHARED_PREFIXES = ("backbone.", "pool.", "pool_norm.")   # identical across arms by design
EXPECTED_EMBEDDING_DIM = 256                            # 2 * nf for both architectures


def count_params(m):
    return int(sum(p.numel() for p in m.parameters()))


model_rows = []
crit = nn.BCEWithLogitsLoss()
for arch in ARCHS:
    ref_state = None
    for arm in ARMS:
        set_all_seeds(SEED)
        model = ECGNet(arch, META[arm].shape[1]).to(DEVICE)
        # initial backbone / pool / pool_norm weights (captured BEFORE any forward pass,
        # which would update BatchNorm running statistics through arm-dependent RNG draws)
        state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()
                 if k.startswith(SHARED_PREFIXES)}
        model.train()
        g = torch.Generator().manual_seed(SEED)
        xb = torch.randn(8, N_LEADS, CROP, generator=g).to(DEVICE)
        mb = torch.randn(8, META[arm].shape[1], generator=g).to(DEVICE)
        yb = torch.randint(0, 2, (8, 5), generator=g).float().to(DEVICE)
        out = model(xb, mb)
        loss = crit(out, yb)
        loss.backward()
        assert tuple(out.shape) == (8, 5), "{} {}: output shape {}".format(arch, arm, tuple(out.shape))
        assert bool(torch.isfinite(loss)), "{} {}: non-finite loss on a random batch".format(arch, arm)
        assert all(p.grad is not None and bool(torch.isfinite(p.grad).all())
                   for p in model.parameters()), "{} {}: missing / non-finite gradients".format(arch, arm)
        assert model.embedding_dim == EXPECTED_EMBEDDING_DIM, \
            "{} {}: embedding_dim {} != {}".format(arch, arm, model.embedding_dim, EXPECTED_EMBEDDING_DIM)
        expected_in = model.embedding_dim + EXPECTED_META_DIM[arm]
        assert model.hidden[0].in_features == expected_in, \
            "{} {}: fusion layer in_features {} != {}".format(arch, arm, model.hidden[0].in_features, expected_in)
        assert model.final.in_features == 128
        # backbone / pool / pool_norm must be initialised identically across arms (same seed)
        if ref_state is None:
            ref_state = state
        else:
            assert state.keys() == ref_state.keys() and all(torch.equal(state[k], ref_state[k]) for k in state), \
                "{} {}: initial backbone/pool_norm weights differ from signal_only at the same seed".format(arch, arm)
        model_rows.append({"arch": arch, "arm": arm, "n_params": count_params(model),
                           "embedding_dim": model.embedding_dim, "meta_dim": EXPECTED_META_DIM[arm],
                           "fusion_layer_in_features": model.hidden[0].in_features,
                           "forward_backward_ok": True, "shared_init_identical_to_signal_only": True})
        del model, out, loss
    torch.cuda.empty_cache()
model_df = pd.DataFrame(model_rows)
print(model_df.to_string(index=False))
print("forward+backward on a random batch (B=8, 12x250): output (8, 5), finite loss and gradients "
      "(loss value not printed); fusion-layer input dims (256 + 0/3/82) and cross-arm identity of initial "
      "backbone/pool_norm weights: PASS")
save_csv(model_df, PREFIX + "model_summary.csv", index=False)


# =====================================================================================
# PART E - TIMING AND VRAM BENCHMARK (training folds 1-8 only; no evaluation)
# =====================================================================================
hdr("PART E - TIMING AND VRAM BENCHMARK (arm signal_plus_all, 1 epoch on folds 1-8)")


class RandomCropDataset(Dataset):
    """Training records -> random 250-sample crop, per-lead normalized with folds 1-8 stats."""

    def __init__(self, X, rows, meta, y, crop, seed):
        self.X, self.rows, self.meta, self.y, self.crop = X, rows, meta, y, crop
        self.rng = np.random.default_rng(seed)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        r = self.rows[i]
        s = int(self.rng.integers(0, N_SAMPLES - self.crop + 1))
        x = (self.X[r, :, s:s + self.crop] - LEAD_MEAN_F32) / LEAD_STD_F32
        return (torch.from_numpy(np.ascontiguousarray(x, dtype=np.float32)),
                torch.from_numpy(self.meta[r]), torch.from_numpy(self.y[r]))


def _is_oom(e):
    return "out of memory" in str(e).lower()


def run_one_epoch(arch, arm, batch_size):
    set_all_seeds(SEED)
    model = ECGNet(arch, META[arm].shape[1]).to(DEVICE)
    ds = RandomCropDataset(X, train_rows, META[arm], Y, CROP, SEED)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=True, num_workers=NUM_WORKERS,
                        pin_memory=True, drop_last=False,
                        generator=torch.Generator().manual_seed(SEED))
    opt = torch.optim.AdamW(model.parameters(), lr=1e-2, weight_decay=WEIGHT_DECAY, betas=(0.9, 0.99))
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=1e-2, total_steps=len(loader))
    model.train()
    all_finite = True
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    for xb, mb, yb in loader:
        xb = xb.to(DEVICE, non_blocking=True)
        mb = mb.to(DEVICE, non_blocking=True)
        yb = yb.to(DEVICE, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        loss = crit(model(xb, mb), yb)
        loss.backward()
        opt.step()
        sched.step()
        all_finite = all_finite and bool(torch.isfinite(loss.detach()))
    torch.cuda.synchronize()
    secs = time.time() - t0
    peak_mb = torch.cuda.max_memory_allocated() / 1024 ** 2
    n_batches = len(loader)
    del model, opt, sched, loader
    torch.cuda.empty_cache()
    return secs, peak_mb, n_batches, all_finite


bench_rows = []
BATCH_SIZE_USED = {}
for arch in ARCHS:
    bs, oom_events = BENCH_BATCH_SIZE, 0
    while True:
        try:
            secs, peak_mb, n_batches, finite = run_one_epoch(arch, "signal_plus_all", bs)
            break
        except RuntimeError as e:
            if not _is_oom(e) or bs <= 1:
                raise
            oom_events += 1
            torch.cuda.empty_cache()
            warn("{}: CUDA OOM at batch size {} -> halving.".format(arch, bs))
            bs //= 2
    assert finite, "{}: non-finite training loss during the benchmark epoch - aborting.".format(arch)
    BATCH_SIZE_USED[arch] = bs
    bench_rows.append({"arch": arch, "arm": "signal_plus_all", "batch_size": bs, "oom_events": oom_events,
                       "n_train_records": len(train_rows), "n_batches": n_batches, "crop_samples": CROP,
                       "seconds_per_epoch": round(secs, 2), "max_memory_allocated_mb": round(peak_mb, 1),
                       "training_loss_finite": finite})
    print("  {:14s} bs={:3d} | {:6.1f} s/epoch | peak VRAM allocated {:7.1f} MB | {} batches | OOM events {}".format(
        arch, bs, secs, peak_mb, n_batches, oom_events))
bench_df = pd.DataFrame(bench_rows)
save_csv(bench_df, PREFIX + "benchmark.csv", index=False)

# --- projected budget ---
N_DEV_RUNS_PER_ARCH = len(ARMS) * N_SEEDS * len(LR_GRID)
N_FINAL_RUNS_PER_ARCH = 2 * N_SEEDS                       # signal_plus_physio, signal_plus_all
FINAL_DATA_FACTOR = EXPECTED_N_RECORDS_FOLDS1TO9 / len(train_rows)   # final run trains on folds 1-9
SEC_PER_EPOCH = {r["arch"]: r["seconds_per_epoch"] for r in bench_rows}

budget_rows = []
for arch in ARCHS:
    s = SEC_PER_EPOCH[arch]
    dev_h = N_DEV_RUNS_PER_ARCH * MAX_EPOCHS * s / 3600
    fin_h = N_FINAL_RUNS_PER_ARCH * MAX_EPOCHS * s * FINAL_DATA_FACTOR / 3600
    budget_rows.append({"arch": arch, "seconds_per_epoch": s, "dev_runs": N_DEV_RUNS_PER_ARCH,
                        "final_runs": N_FINAL_RUNS_PER_ARCH, "max_epochs": MAX_EPOCHS,
                        "dev_hours": round(dev_h, 2), "final_hours": round(fin_h, 2),
                        "total_hours": round(dev_h + fin_h, 2)})
budget_df = pd.DataFrame(budget_rows)
TOTAL_HOURS = float(budget_df["total_hours"].sum())
_tot_row = {"arch": "__TOTAL__", "seconds_per_epoch": np.nan, "dev_runs": int(budget_df["dev_runs"].sum()),
            "final_runs": int(budget_df["final_runs"].sum()), "max_epochs": MAX_EPOCHS,
            "dev_hours": round(float(budget_df["dev_hours"].sum()), 2),
            "final_hours": round(float(budget_df["final_hours"].sum()), 2), "total_hours": round(TOTAL_HOURS, 2)}
budget_df = pd.concat([budget_df, pd.DataFrame([_tot_row])], ignore_index=True)
BUDGET_VERDICT = "PASS" if TOTAL_HOURS <= GPU_BUDGET_HOURS else "OVER"
print("\nprojected budget: runs = 2 archs x 3 arms x 3 seeds x {} lr (development) + 2 archs x 2 arms x "
      "3 seeds (final); hours = runs x {} epochs x s/epoch / 3600 (final runs scaled by {:.3f} for "
      "folds 1-9). Evaluation passes on fold 9 are NOT included.".format(len(LR_GRID), MAX_EPOCHS, FINAL_DATA_FACTOR))
print(budget_df.to_string(index=False))
print("TOTAL {:.2f} GPU-hours vs budget {:.0f} -> {}".format(TOTAL_HOURS, GPU_BUDGET_HOURS, BUDGET_VERDICT))
BUDGET_SUGGESTION = None
if BUDGET_VERDICT == "OVER":
    sum_s = sum(SEC_PER_EPOCH.values())
    budget_s = GPU_BUDGET_HOURS * 3600
    final_s = N_FINAL_RUNS_PER_ARCH * MAX_EPOCHS * sum_s * FINAL_DATA_FACTOR
    max_grid = int(math.floor((budget_s - final_s) / (len(ARMS) * N_SEEDS * MAX_EPOCHS * sum_s))) \
        if budget_s > final_s else 0
    max_epochs = int(math.floor(budget_s / (sum_s * (N_DEV_RUNS_PER_ARCH + N_FINAL_RUNS_PER_ARCH * FINAL_DATA_FACTOR))))
    BUDGET_SUGGESTION = ("to fit {:.0f} GPU-hours: EITHER tuning grid size <= {} (with max_epochs={}) OR "
                         "max_epochs <= {} (with grid size {}). Part F is NOT changed by this code.".format(
                             GPU_BUDGET_HOURS, max_grid, MAX_EPOCHS, max_epochs, len(LR_GRID)))
    warn("budget OVER: " + BUDGET_SUGGESTION)
note("the development formula above is the conservative one from the prompt; under the Part F "
     "selection rule only signal_plus_physio (seed 42) needs the full lr grid, so the real "
     "development cost is lower.")
save_csv(budget_df, PREFIX + "budget_projection.csv", index=False)


# =====================================================================================
# PART F - DEVELOPMENT PROTOCOL (write-once, BEFORE any signal-model result exists)
# =====================================================================================
hdr("PART F - DEVELOPMENT PROTOCOL")

if DEV_PROTOCOL_EXISTS:
    print("dev_protocol.json already exists and was verified at STEP 0 (self_sha256={}) -> "
          "Part F SKIPPED, nothing written.".format(DEV_PROTOCOL_SHA))
elif BUDGET_VERDICT != "PASS":
    warn("budget OVER (conservative formula) -> dev_protocol.json NOT written. "
         "Decide grid size / epochs with the reviewer, then re-run v0.4.")
else:
    eval_window_starts = list(range(0, N_SAMPLES - CROP + 1, EVAL_STRIDE))
    dev_protocol = {
        "document": "development protocol for H3 (weeks 4-5)",
        "timestamp": datetime.now().astimezone().isoformat(),
        "code_version": CODE_VERSION,
        "parent_seal_self_sha256": EXPECTED_SEAL_SHA256,
        "parent_seal_path": SEAL_PATH,
        "input_files_sha256": INPUT_HASHES,
        "written_before_any_signal_model_result": True,
        "training_recipe": {
            "source": "reference defaults (Strodthoff et al., ecg_ptbxl_benchmarking)",
            "optimizer": "AdamW",
            "weight_decay": WEIGHT_DECAY,
            "lr_schedule": "one-cycle",
            "batch_size": BATCH_SIZE_USED,
            "batch_size_source": "Part E benchmark of v0.4 (reference default 128; halved only on OOM)",
            "train_input": "random {}-sample crops ({} s at {} Hz)".format(CROP, CROP / FS, FS),
            "evaluation": ("sliding windows of {} samples, stride {} (window starts {}); per-record "
                           "probability = mean over windows".format(CROP, EVAL_STRIDE, eval_window_starts)),
            "epochs": MAX_EPOCHS,
            "epochs_note": ("fixed for every development and final run; the one-cycle schedule spans all "
                            "{} epochs; no epoch selection, no early stopping, no checkpoint "
                            "selection".format(MAX_EPOCHS)),
            "loss": "BCEWithLogits over 5 superclasses (NORM, MI, STTC, CD, HYP)",
        },
        "normalization": "per-lead mean/std from training folds only",
        "tuning_grid": {"learning_rate": LR_GRID},
        "tuning_note": "learning rate is the only tuned hyperparameter -- nothing else is tuned",
        "selection_rule": (
            "Only the learning rate is tuned. lr is chosen ONLY by the fold-9 macro-AUROC, computed once at "
            "the end of epoch 50, of the signal_plus_physio arm, seed 42, per architecture; the chosen lr is "
            "applied IDENTICALLY to signal_only and signal_plus_all. No setting may be chosen, changed, or "
            "re-run based on the between-arm delta. The selected lr is recorded in a later write-once file "
            "before the final fold 1-9 run."),
        "pipeline_sanity_gate": (
            "signal_only (resnet1d_wang, selected settings, seed 42) must reach fold-9 macro-AUROC >= 0.90 "
            "before any between-arm comparison is computed; if it fails, stop and debug, and report the "
            "failure -- do not tune further to pass it."),
        "metadata_fusion": METADATA_FUSION_DESCRIPTION,
        "evaluation_aggregation_note": (
            "per-record probability = MEAN over sliding windows, chosen a priori; the reference "
            "implementation may use max -- stated deviation, not to be changed after results are seen"),
        "status": ("signal_only and all fold-9 comparisons are EXPLORATORY / development; only the week-8 "
                   "fold-10 analysis is confirmatory."),
        "self_sha256_procedure": SELF_SHA256_PROCEDURE,
        "write_once": True,
    }
    dev_protocol = json.loads(json.dumps(dev_protocol, ensure_ascii=False, default=_json_default))
    assert "self_sha256" not in dev_protocol
    DEV_PROTOCOL_SHA = hashlib.sha256(canonical_json_bytes(dev_protocol)).hexdigest()
    dev_protocol["self_sha256"] = DEV_PROTOCOL_SHA
    with open(DEV_PROTOCOL_PATH, "x", encoding="utf-8") as f:     # refuses to overwrite
        json.dump(dev_protocol, f, ensure_ascii=False, indent=2)
    _s, _r, _ = verify_self_sha256(DEV_PROTOCOL_PATH)
    assert _s == _r == DEV_PROTOCOL_SHA, "dev_protocol.json self_sha256 verification FAILED."
    print("[saved] " + DEV_PROTOCOL_PATH)
    print("dev_protocol self_sha256 (verified from disk): " + DEV_PROTOCOL_SHA)
    print(json.dumps(dev_protocol, ensure_ascii=False, indent=2))
OUTPUT_FILES.append(DEV_PROTOCOL_PATH)


# =====================================================================================
# MANIFEST (hashes of all outputs except itself)
# =====================================================================================
hdr("MANIFEST")
manifest = {
    "code_version": CODE_VERSION,
    "run_started": RUN_STARTED,
    "run_finished": datetime.now().astimezone().isoformat(),
    "dataset_root": DATA_ROOT,
    "output_dir": OUT_DIR,
    "input_files_sha256": INPUT_HASHES,
    "parent_seal_self_sha256": EXPECTED_SEAL_SHA256,
    "dev_protocol_self_sha256": DEV_PROTOCOL_SHA,
    "dev_protocol_written_this_run": not DEV_PROTOCOL_EXISTS,
    "split_protocol": {
        "folds_loaded": "1-9", "fold10_opened": False,
        "n_fold10_records_in_source_never_opened": n_fold10_in_source,
        "n_train_records": len(train_rows), "n_eval_records": len(eval_rows),
        "n_train_patients": len(train_patients), "n_eval_patients": len(eval_patients),
        "n_record_files_opened_this_run": N_FILES_OPENED, "signal_cache_reused": cache_ok,
    },
    "lead_order": LEAD_NAMES,
    "environment": ENV,
    "metadata_arms": {arm: ARM_COLUMNS[arm] for arm in ARMS},
    "metadata_scaling": "StandardScaler on age_imputed only, fit on folds 1-8; all other columns raw 0/1",
    "final_run_refit_note": FINAL_RUN_REFIT_NOTE,
    "metadata_fusion_implementation": METADATA_FUSION_DESCRIPTION,
    "model_summary": model_rows,
    "benchmark": bench_rows,
    "budget": {"rows": budget_rows, "total_hours": TOTAL_HOURS, "budget_hours": GPU_BUDGET_HOURS,
               "verdict": BUDGET_VERDICT, "suggestion": BUDGET_SUGGESTION},
    "no_performance_metric_computed": True,
    "warnings": WARNINGS,
    "outputs": [],
}
for p in OUTPUT_FILES:
    manifest["outputs"].append({"file": p, "bytes": os.path.getsize(p), "sha256": sha256_of(p)})
MANIFEST_PATH = os.path.join(OUT_DIR, PREFIX + "manifest.json")
with open(MANIFEST_PATH, "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2, default=_json_default)
for o in manifest["outputs"]:
    print("  {}  {}".format(o["sha256"], o["file"]))
print("[saved] " + MANIFEST_PATH)
print("manifest sha256:", sha256_of(MANIFEST_PATH))


# =====================================================================================
# FINAL SUMMARY
# =====================================================================================
hdr("SUMMARY - " + CODE_VERSION)
print("seal                       : verified ({})".format(EXPECTED_SEAL_SHA256))
print("gpu                        : {} ({} GB) | torch {} | CUDA {} | cuDNN {}".format(
    ENV["gpu_name"], ENV["gpu_total_vram_gb"], ENV["torch"], ENV["torch_cuda"], ENV["cudnn"]))
print("signals                    : {} records x 12 x 1000 (folds 1-9), NaN/Inf 0, flat leads {}".format(
    N_REC, int(_tot["n_flat_leads"])))
for r in bench_rows:
    print("benchmark {:16s} : bs {} | {:.1f} s/epoch | {:.0f} MB peak".format(
        r["arch"], r["batch_size"], r["seconds_per_epoch"], r["max_memory_allocated_mb"]))
print("budget                     : {:.2f} / {:.0f} GPU-hours -> {}".format(TOTAL_HOURS, GPU_BUDGET_HOURS, BUDGET_VERDICT))
print("dev_protocol               : {} ({})".format(
    "written this run" if not DEV_PROTOCOL_EXISTS else "already existed, skipped", DEV_PROTOCOL_SHA))
print("\nwarnings ({}):".format(len(WARNINGS)))
for w in WARNINGS:
    print("  - " + w)
print("\nNo performance metric was computed in v0.4. Fold 10 was not opened.")


STEP 0 - SEAL INTEGRITY
pre_registration.json stored self_sha256    : 5183f54569f3358a5339b16a2c3c3be49f5c7798eabe6b0bb25946119c2ba469
pre_registration.json recomputed self_sha256: 5183f54569f3358a5339b16a2c3c3be49f5c7798eabe6b0bb25946119c2ba469
expected (v0.3 output)                      : 5183f54569f3358a5339b16a2c3c3be49f5c7798eabe6b0bb25946119c2ba469
seal integrity: PASS

--- H3_protocol (from the sealed file; it governs everything below) ---
{
  "comparison": "signal_plus_physio [age, sex] vs signal_plus_all [age, sex, device, site, nurse]",
  "metadata_fusion": "metadata vector, encoded exactly as in v0.2.1 (encoder fit on training folds only), concatenated to the network's global-pooled embedding before the final linear classification head",
  "primary_architecture": "resnet1d_wang on 100 Hz",
  "robustness_architecture": "inception1d",
  "architecture_rule": "disagreement is reported, never resolved by selection",
  "development": "weeks 4-5: train folds 1-8, validate fold 9; 